In [ ]:
# Cell 1: Trinity Engine
import pandas as pd
import numpy as np
from scipy.interpolate import PchipInterpolator, CubicSpline, Akima1DInterpolator

train = pd.read_csv("train_dataset.csv")
test = pd.read_csv("test_dataset.csv")
test['implied_volatility'] = np.nan
df = pd.concat([train, test], ignore_index=True)

df['pred_pchip'] = np.nan
df['pred_spline'] = np.nan
df['pred_akima'] = np.nan

grouped = df.groupby(['parsed_dt', 'option_type', 'expiry_str'])

for _, group in grouped:
    known = group[group['implied_volatility'].notna()].copy()
    unknown = group[group['implied_volatility'].isna()].copy()
    if unknown.empty or len(known) < 2: continue
        
    known = known.sort_values('moneyness')
    x_train, y_train = known['moneyness'].values, known['implied_volatility'].values
    x_test = unknown['moneyness'].values
    
    # Kernel 1: PCHIP
    pchip = PchipInterpolator(x_train, y_train, extrapolate=True)
    df.loc[unknown.index, 'pred_pchip'] = pchip(x_test)
    
    # Kernel 2: Cubic Spline
    if len(known) >= 4:
        spline = CubicSpline(x_train, y_train, bc_type='natural', extrapolate=True)
        df.loc[unknown.index, 'pred_spline'] = spline(x_test)
    else:
        df.loc[unknown.index, 'pred_spline'] = pchip(x_test)
            
    # Kernel 3: Akima
    if len(known) >= 3:
        akima = Akima1DInterpolator(x_train, y_train)
        df.loc[unknown.index, 'pred_akima'] = akima(x_test)
    else:
        df.loc[unknown.index, 'pred_akima'] = pchip(x_test)

df['final_value'] = (df['pred_spline'] * 0.25) + (df['pred_pchip'] * 0.35) + (df['pred_akima'] * 0.40)
df.to_csv("filled_dataset.csv", index=False)